# Who are my guests? Clustering with k-means

UL1 of *Predictive Analytics*. Work in **pairs**; swap at every station: **predict, run, change, decide.**

**Read** questions are in the style of the exam: answer them on paper first, then run the cell below.

**Stuck?** Every station only needs the *Setup* cell. Just go on with the next station.

| Station | Question | Time |
|---|---|---|
| 1 | Distance and scaling | 10 min |
| 2 | k-means on real guests | 12 min |
| 3 | How many groups? | 12 min |
| 4 | Name the groups | 11 min |
| Extra | For the fast ones | |

> *From: Mara, hotel manager. Here are 5,000 of our bookings. Please sort the guests into groups, without looking at who cancelled. I want to write to every group in its own way.*

Data: Antonio, Almeida, Nunes 2019, [Hotel booking demand datasets](https://doi.org/10.1016/j.dib.2018.11.126), CC BY 4.0, cleaned, a random sample. Every booking counts as one guest. The six features: `lead_time` (days between booking and arrival), `price` (per night), `nights`, `adults`, `kids`, `earlier_stays` (earlier bookings of the guest that were not cancelled).

## Setup

Run this cell once (about 5 seconds). You do not have to read the code.

In [ ]:
#@title Setup (run me) { display-mode: "form" }
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.preprocessing import StandardScaler

URL = "https://fjoelsak.github.io/thm_wk-1609-predictiveanalytics/cl1-who-gets-the-offer/hotel-bookings.csv"
FILE = Path("hotel-bookings.csv")
df = pd.read_csv(FILE if FILE.exists() else URL)

# 5,000 random bookings are our guests. "cancelled" is NOT used for the groups, only to look at them afterwards.
guests = df.sample(5000, random_state=0).reset_index(drop=True)
guests["nights"] = guests["weekend_nights"] + guests["week_nights"]
guests["kids"] = guests["children"] + guests["babies"]
FEATURES = ["lead_time", "price", "nights", "adults", "kids", "earlier_stays"]


def standardise(features):
    """z = (x - mean) / standard deviation for every column."""
    return pd.DataFrame(StandardScaler().fit_transform(guests[features]), columns=features, index=guests.index)


def group_guests(features=FEATURES, k=5, scale=True, n_init=10, seed=0, init="k-means++"):
    """k-means on the guests; returns the fitted model (km.labels_ = group of every guest, km.inertia_ = SSE)."""
    X = standardise(features) if scale else guests[features]
    return KMeans(n_clusters=k, n_init=n_init, random_state=seed, init=init).fit(X)


def distances(table):
    """Euclidean distance of every pair of the guests 1, 2 and 18."""
    return {f"{a} - {b}": round(float(np.linalg.norm(table.loc[a] - table.loc[b])), 2) for a, b in [(1, 2), (1, 18), (2, 18)]}


def profile(labels, features=FEATURES, cancelled=False):
    """Number of guests and the mean of every feature per group (optionally the share of cancellations)."""
    table = guests[features].groupby(labels).mean().round(2)
    table.insert(0, "guests", pd.Series(labels).value_counts().sort_index())
    if cancelled:
        table["cancelled"] = guests["cancelled"].groupby(labels).mean().round(2)
    return table


print(f"{len(guests)} guests, features: {', '.join(FEATURES)}")
guests[FEATURES].describe().round(1)

## Station 1: Distance and scaling

Three guests of the table: number 1, 2 and 18, with the price per night and the number of nights.

**Predict:** which two of the three guests are closest to each other? And will that change if we standardise both features first?

**Prediction:** *write here*

In [ ]:
three = guests.loc[[1, 2, 18], ["price", "nights"]]
print(three)

print("in € and nights: ", distances(three))
z = standardise(["price", "nights"]).loc[[1, 2, 18]]   # mean and standard deviation of ALL 5,000 guests
print("standardised:    ", distances(z))

**Change:** add `lead_time` (in days) as a third feature. Which pair is closest now, without and with standardising?

In [ ]:
features = ["price", "nights"]   # change me, e.g. ["price", "nights", "lead_time"]

raw = guests.loc[[1, 2, 18], features]
print("in original units:", distances(raw))
print("standardised:     ", distances(standardise(features).loc[[1, 2, 18]]))

**Read:** what does the code print?

```python
a = np.array([100, 2])
b = np.array([150, 3])
print(round(np.sqrt(((a - b) ** 2).sum()), 2))
```

**Answer:** *write here*

In [ ]:
a = np.array([100, 2])
b = np.array([150, 3])
print(round(np.sqrt(((a - b) ** 2).sum()), 2))

**Decide:** give one reason why we standardise the features before k-means.

**Answer:** *write here*

## Station 2: k-means on real guests

**Predict:** which kinds of guests will five groups show? Write down two or three before you run anything.

**Prediction:** *write here*

In [ ]:
km = group_guests(k=5)
print("SSE:", round(km.inertia_))
profile(km.labels_)

The groups as a picture (two of the six features):

In [ ]:
plt.scatter(guests["lead_time"], guests["price"], c=km.labels_, cmap="viridis", s=8, alpha=0.6)
plt.xlabel("lead time in days")
plt.ylabel("price per night in €")
plt.show()

**Change:** change `k`, the features or switch `scale` off. What happens to the groups?

In [ ]:
km = group_guests(k=5, scale=True)   # change me: k, features=[...], scale=False
print("SSE:", round(km.inertia_))
profile(km.labels_)

**Read:** what does the code print?

```python
points = np.array([[1, 2], [2, 1], [5, 4]])
centre = points[:2].mean(axis=0)
print(centre, ((points[:2] - centre) ** 2).sum())
```

**Answer:** *write here*

In [ ]:
points = np.array([[1, 2], [2, 1], [5, 4]])
centre = points[:2].mean(axis=0)
print(centre, ((points[:2] - centre) ** 2).sum())

**Decide:** which of the groups surprises you most, and why?

**Answer:** *write here*

## Station 3: How many groups?

**Predict:** the SSE for k = 1, 2, 3 … 10. Will it reach 0 somewhere? Where do you expect the elbow?

**Prediction:** *write here*

In [ ]:
ks = range(1, 11)
sse = [group_guests(k=k).inertia_ for k in ks]

plt.plot(ks, sse, color="green", linewidth=3, marker="o")
plt.xlabel("number of groups k")
plt.ylabel("SSE")
plt.show()
print("gain compared with k - 1:", [round(a - b) for a, b in zip(sse, sse[1:])])

**The start matters:** ten runs with one random start each (`n_init=1`). How different are the results?

In [ ]:
best = group_guests(k=5)
for seed in range(10):
    one = group_guests(k=5, n_init=1, seed=seed, init="random")
    print(f"start {seed}: SSE {one.inertia_:7.0f}   agreement with the best run {adjusted_rand_score(best.labels_, one.labels_):.2f}")

**Change:** set `n_init` to 1 and to 10 for the same `seed`. Which is more reliable, and why?

In [ ]:
for n_init in (1, 10):
    print(n_init, round(group_guests(k=5, n_init=n_init, seed=3, init="random").inertia_))  # change me

**Read:** after standardising, every feature has the mean 0 and the standard deviation 1. What does the code print?

```python
X = np.array([[-1.0, 1.0], [1.0, -1.0], [-1.0, -1.0], [1.0, 1.0]])
print(KMeans(n_clusters=1, n_init=1).fit(X).inertia_)
```

**Answer:** *write here*

In [ ]:
X = np.array([[-1.0, 1.0], [1.0, -1.0], [-1.0, -1.0], [1.0, 1.0]])
print(KMeans(n_clusters=1, n_init=1).fit(X).inertia_)

**Decide:** how many groups do you take for Mara? Give one reason that is not only the elbow.

**Answer:** *write here*

## Station 4: Name the groups

The groups were found **without** the cancellations. Now we look at them.

**Predict:** which group has the highest share of cancellations, which the lowest?

**Prediction:** *write here*

In [ ]:
km = group_guests(k=5)
table = profile(km.labels_, cancelled=True)
table

In [ ]:
table["cancelled"].plot.bar(color="green")
plt.xlabel("group")
plt.ylabel("share of cancellations")
plt.show()

**Change:** give every group a name and write one sentence for Mara: what would you do for this group?

In [ ]:
names = {0: "?", 1: "?", 2: "?", 3: "?", 4: "?"}   # change me, e.g. 0: "families"
idea = "?"                                         # one sentence for Mara

table.index = [names[i] for i in table.index]
print(idea)
table

**Decide:** which group would you write to first, and what would you say?

**Answer:** *write here*

**Hand in:** your names of the groups and the one sentence for Mara.

----
## Extra: for the fast ones

1. **Outliers:** the 19 regulars have 23 earlier stays on average. Replace `earlier_stays` by `np.log1p(earlier_stays)` in `guests` and cluster again. What happens to the groups?
2. **Another sample:** change `random_state` of the sample in the setup cell. Do you get the same five groups?
3. **Exam question:** why is the centre of a group in general not one of its guests?

In [ ]:
# your space